In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import arff

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Read Dataset
file_path = '/kaggle/input/datasets/muratkokludataset/dry-bean-dataset/Dry_Bean_Dataset/Dry_Bean_Dataset.arff'

data, meta = arff.loadarff(file_path)

df = pd.DataFrame(data)

for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.decode('utf-8')

print(df.head())
print("Dataset Shape:", df.shape)


In [ ]:

X = df.drop('Class', axis=1).values
y = df['Class'].values

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)


In [ ]:

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print("Classes:", label_encoder.classes_)
print("Encoded Classes:", np.unique(y))


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)


In [ ]:

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Scaling Completed!")


In [ ]:

num_classes = len(np.unique(y_train))

Y_train = np.eye(num_classes)[y_train]

print("Number of Classes:", num_classes)
print("Y_train Shape:", Y_train.shape)


In [ ]:

np.random.seed(42)

n_samples, n_features = X_train.shape

W = np.random.randn(n_features, num_classes) * 0.01
b = np.zeros((1, num_classes))

print("Weights Shape:", W.shape)
print("Bias Shape:", b.shape)


In [ ]:

def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


def compute_loss(Y_true, Y_pred):
    epsilon = 1e-15
    Y_pred = np.clip(Y_pred, epsilon, 1 - epsilon)

    loss = -np.mean(
        np.sum(Y_true * np.log(Y_pred), axis=1)
    )

    return loss


In [ ]:

# Hyperparameters

learning_rate = 0.001
epochs = 1000

rho = 0.9
beta = 0.9
epsilon = 1e-8

# Initialize RMSProp variables

cache_W = np.zeros_like(W)
cache_b = np.zeros_like(b)

# Initialize Momentum variables

vW = np.zeros_like(W)
vb = np.zeros_like(b)

loss_history = []

# Training Loop

for epoch in range(epochs):

    # Forward Propagation
    z = np.dot(X_train, W) + b
    y_pred = softmax(z)

    # Calculate Loss
    loss = compute_loss(Y_train, y_pred)
    loss_history.append(loss)

    # Calculate Gradients
    error = y_pred - Y_train

    dW = np.dot(X_train.T, error) / n_samples
    db = np.sum(error, axis=0, keepdims=True) / n_samples

    # RMSProp: Update squared gradient averages
    cache_W = rho * cache_W + (1 - rho) * (dW ** 2)
    cache_b = rho * cache_b + (1 - rho) * (db ** 2)

    # Normalize Gradients
    rms_dW = dW / (np.sqrt(cache_W) + epsilon)
    rms_db = db / (np.sqrt(cache_b) + epsilon)

    # Momentum Update
    vW = beta * vW + rms_dW
    vb = beta * vb + rms_db

    # Update Weights and Bias
    W -= learning_rate * vW
    b -= learning_rate * vb

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

print("RMSProp with Momentum Training Completed!")


In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(loss_history)

plt.title('RMSProp with Momentum - Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Categorical Cross-Entropy Loss')

plt.grid(True)
plt.show()


In [ ]:

z_test = np.dot(X_test, W) + b

y_test_prob = softmax(z_test)

y_pred = np.argmax(y_test_prob, axis=1)

print("Predicted:", y_pred[:10])
print("Actual:", y_test[:10])


In [ ]:

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test, y_pred, average='macro'
)

recall = recall_score(
    y_test, y_pred, average='macro'
)

f1 = f1_score(
    y_test, y_pred, average='macro'
)

print("RMSProp with Momentum Results")
print("-----------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")


In [ ]:

print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))


In [ ]:

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

fig, ax = plt.subplots(figsize=(10, 8))

disp.plot(
    cmap='Blues',
    xticks_rotation=45,
    ax=ax
)

plt.title('RMSProp with Momentum - Confusion Matrix')
plt.show()


In [ ]:

rmsprop_results = {
    'Optimizer': 'RMSProp with Momentum',
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1,
    'Loss': loss_history[-1]
}

results_df = pd.DataFrame([rmsprop_results])

print(results_df)
